# InsPLAD LVLM × Vision Expert resumable smoke

이 노트북은 기존 vision-expert smoke 실험을 중간 중단에 강하게 다시 구성한 버전입니다.

- `런타임 → 모두 실행`으로 시작합니다.
- Colab 연결이 끊기거나 셀이 실패하면 새 런타임에서 다시 `모두 실행`하면 됩니다.
- 완료된 단계는 결과 파일을 실제로 검증한 뒤 자동으로 건너뜁니다.
- 로그, 상태, ROI, 예측 JSONL, leaderboard는 Google Drive에 즉시 저장됩니다.
- PatchCore는 asset별로 이어서 실행합니다.
- AnomalyCLIP은 이미지별로 이어서 실행합니다.
- Qwen / InternVL은 예측 1건마다 원자적으로 저장해 다음 실행에서 완료된 응답을 재사용합니다.
- Dinomaly가 학습 도중 끊기면 Dinomaly 단계만 다시 학습합니다. 이미 끝난 다른 단계는 다시 실행하지 않습니다.

필요한 Colab Secret:

- `GITHUB_TOKEN`: 필수
- `HF_TOKEN`: 권장
- `LUXIA_API_KEY`: 이번 실험에서는 사용하지 않음

Google Drive에 필요한 파일:

- `unsupervised_anomaly_detection.zip`
- `supervised_fault_classification.zip`

기본 결과 위치:

```text
MyDrive/lvlm_insplad_results/vision_expert_smoke_v1/
```

In [ ]:
# 0) 실험 설정 — 보통 이 셀은 수정하지 않아도 됩니다.
RUN_FOLDER = "vision_expert_smoke_v1"
SMOKE_SIZE = 50
SEED = 42

MODELS_TO_RUN = ["qwen3_5_4b", "internvl3_5_8b"]
EXPERTS_TO_RUN = ["patchcore", "dinomaly", "anomalyclip"]

MAX_TRAIN_NORMALS_PER_ASSET = 200
PATCHCORE_CORESET_RATIO = 0.05

DINOMALY_ENCODER = "vit_small_patch14_reg4_dinov2"
DINOMALY_DECODER_DEPTH = 8
DINOMALY_MAX_STEPS = 300
DINOMALY_BATCH_CANDIDATES = [8, 4, 2]

# LVLM은 매 응답마다 Drive에 저장합니다. 가장 안전하지만 파일 쓰기가 조금 늘어납니다.
LVLM_CHECKPOINT_EVERY = 1
LVLM_RETRY_ATTEMPTS = 3

RUN_REFERENCE = True
RUN_ROI_LVLM = True
RUN_COMBINED = True
SHOW_ROI_PREVIEW = True
AUTO_DOWNLOAD_RESULT_ZIP = False  # 전체 결과 ZIP은 클 수 있어 기본 자동 다운로드하지 않습니다.
AUTO_DOWNLOAD_REVIEW_ZIP = True   # ChatGPT에 올릴 작은 결과 요약 ZIP은 자동 다운로드합니다.
SAVE_FULL_RESULT_ZIP_TO_DRIVE = False  # 전체 workspace까지 묶으려면 True (용량 큼)

# ZIP이 MyDrive 여러 위치에 중복될 때만 폴더를 지정하세요.
# 예: DRIVE_DATA_DIR = "datasets/InsPLAD"
DRIVE_DATA_DIR = None

# 특정 단계만 강제 재실행할 때 이름을 넣습니다.
# 가능한 값:
# prepare_qa, prepare_workspace, patchcore_roi, dinomaly_roi,
# anomalyclip_roi, roi_audit, reference_inference,
# roi_inference, select_winner, combined_inference, finalize
FORCE_STAGES = set()

REPO_REF = "main"

assert 20 <= SMOKE_SIZE <= 50
assert set(MODELS_TO_RUN) <= {"qwen3_5_4b", "internvl3_5_8b"}
assert set(EXPERTS_TO_RUN) <= {"patchcore", "dinomaly", "anomalyclip"}
assert LVLM_CHECKPOINT_EVERY > 0
assert LVLM_RETRY_ATTEMPTS > 0

print({
    "run_folder": RUN_FOLDER,
    "smoke_size": SMOKE_SIZE,
    "models": MODELS_TO_RUN,
    "experts": EXPERTS_TO_RUN,
    "force_stages": sorted(FORCE_STAGES),
})

In [ ]:
# 1) Drive, 상태 기록, 저장소, 의존성
import csv
import gc
import hashlib
import importlib
import importlib.metadata
import json
import os
import re
import shutil
import subprocess
import sys
import time
import traceback
import uuid
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

if not Path("/content").exists():
    raise RuntimeError("Google Colab에서 실행하세요.")

from google.colab import drive, userdata
drive.mount("/content/drive", force_remount=False)

DRIVE_ROOT = Path("/content/drive/MyDrive")
BASE_RESULT_ROOT = DRIVE_ROOT / "lvlm_insplad_results"
BASE_RESULT_ROOT.mkdir(parents=True, exist_ok=True)

EXPERIMENT_IDENTITY = {
    "schema_version": 2,
    "run_folder": RUN_FOLDER,
    "smoke_size": SMOKE_SIZE,
    "seed": SEED,
    "models": MODELS_TO_RUN,
    "experts": EXPERTS_TO_RUN,
    "max_train_normals_per_asset": MAX_TRAIN_NORMALS_PER_ASSET,
    "patchcore_coreset_ratio": PATCHCORE_CORESET_RATIO,
    "dinomaly_encoder": DINOMALY_ENCODER,
    "dinomaly_decoder_depth": DINOMALY_DECODER_DEPTH,
    "dinomaly_max_steps": DINOMALY_MAX_STEPS,
}
EXPERIMENT_HASH = hashlib.sha256(
    json.dumps(EXPERIMENT_IDENTITY, sort_keys=True).encode("utf-8")
).hexdigest()

candidate_root = BASE_RESULT_ROOT / RUN_FOLDER
candidate_config = candidate_root / "run_config_resumable.json"
if candidate_config.is_file():
    try:
        previous_config = json.loads(candidate_config.read_text(encoding="utf-8"))
    except Exception:
        previous_config = {}
    previous_hash = str(previous_config.get("experiment_hash", ""))
    if previous_hash and previous_hash != EXPERIMENT_HASH:
        RUN_ROOT = BASE_RESULT_ROOT / f"{RUN_FOLDER}__{EXPERIMENT_HASH[:8]}"
        print("기존 폴더의 설정이 달라 별도 폴더를 사용합니다:", RUN_ROOT)
    else:
        RUN_ROOT = candidate_root
else:
    # 기존 v1 결과 폴더가 있으면 그대로 채택해 재사용합니다.
    RUN_ROOT = candidate_root

LOG_ROOT = RUN_ROOT / "logs"
STATE_PATH = RUN_ROOT / "run_state.json"
EVENTS_PATH = LOG_ROOT / "events.jsonl"
RUN_LOG_PATH = LOG_ROOT / "run.log"
RUN_ROOT.mkdir(parents=True, exist_ok=True)
LOG_ROOT.mkdir(parents=True, exist_ok=True)

def utc_now():
    return datetime.now(timezone.utc).isoformat()

def atomic_write_text(path, text):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(f".{path.name}.{uuid.uuid4().hex}.tmp")
    with tmp.open("w", encoding="utf-8") as handle:
        handle.write(text)
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass
    tmp.replace(path)

def atomic_write_json(path, value):
    atomic_write_text(
        path,
        json.dumps(value, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
    )

def read_json(path, default=None):
    path = Path(path)
    if not path.is_file():
        return {} if default is None else default
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return {} if default is None else default

def append_event(event, **payload):
    row = {"time_utc": utc_now(), "event": event, **payload}
    EVENTS_PATH.parent.mkdir(parents=True, exist_ok=True)
    with EVENTS_PATH.open("a", encoding="utf-8") as handle:
        handle.write(json.dumps(row, ensure_ascii=False) + "\n")
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass

STATE = read_json(
    STATE_PATH,
    {
        "schema_version": 2,
        "experiment_hash": EXPERIMENT_HASH,
        "run_root": str(RUN_ROOT),
        "stages": {},
        "sessions": [],
    },
)
STATE.setdefault("stages", {})
STATE.setdefault("sessions", [])
SESSION_ID = f"{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}-{uuid.uuid4().hex[:8]}"
STATE["sessions"].append({
    "session_id": SESSION_ID,
    "started_at_utc": utc_now(),
})
STATE["sessions"] = STATE["sessions"][-50:]
atomic_write_json(STATE_PATH, STATE)

def mark_stage(name, status, **details):
    stage = dict(STATE["stages"].get(name, {}))
    stage.update(details)
    stage["status"] = status
    stage["updated_at_utc"] = utc_now()
    if status == "running":
        stage["started_at_utc"] = utc_now()
        stage["session_id"] = SESSION_ID
        stage.pop("error", None)
    if status == "complete":
        stage["completed_at_utc"] = utc_now()
    STATE["stages"][name] = stage
    atomic_write_json(STATE_PATH, STATE)
    append_event("stage_status", stage=name, status=status, **details)

def stage_is_complete(name, validator):
    if name in FORCE_STAGES:
        return False
    try:
        return bool(validator())
    except Exception as exc:
        append_event(
            "stage_validation_failed",
            stage=name,
            error=f"{type(exc).__name__}: {exc}",
        )
        return False

def run_stage(name, action, validator):
    if stage_is_complete(name, validator):
        mark_stage(name, "complete", reused=True)
        print(f"[RESUME] {name}: 기존 결과 검증 완료 → 건너뜀")
        return

    mark_stage(name, "running", reused=False)
    print(f"\n[START] {name}")
    try:
        action()
        if not validator():
            raise RuntimeError(f"{name} 실행 후 결과 검증에 실패했습니다.")
    except KeyboardInterrupt:
        mark_stage(name, "interrupted", error="KeyboardInterrupt")
        raise
    except BaseException as exc:
        error = f"{type(exc).__name__}: {exc}\n{traceback.format_exc(limit=12)}"
        mark_stage(name, "failed", error=error)
        raise
    else:
        mark_stage(name, "complete", reused=False)
        print(f"[DONE] {name}")

SENSITIVE_VALUES = []

def redact_text(value):
    text = str(value)
    text = re.sub(
        r"x-access-token:[^@\s]+@",
        "x-access-token:***@",
        text,
    )
    for secret in SENSITIVE_VALUES:
        if secret:
            text = text.replace(secret, "***")
    return text

def run_cmd(cmd, *, cwd=None, env=None, log_name="subprocess"):
    cmd = [str(value) for value in cmd]
    command_text = redact_text(" ".join(cmd))
    append_event("command_start", name=log_name, command=command_text)
    log_path = LOG_ROOT / f"{log_name}.log"
    merged_env = dict(os.environ)
    if env:
        merged_env.update(env)
    with log_path.open("a", encoding="utf-8") as log_handle, RUN_LOG_PATH.open(
        "a", encoding="utf-8"
    ) as run_handle:
        header = f"\n[{utc_now()}] $ {command_text}\n"
        log_handle.write(header)
        run_handle.write(header)
        process = subprocess.Popen(
            cmd,
            cwd=str(cwd) if cwd else None,
            env=merged_env,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            safe_line = redact_text(line)
            print(safe_line, end="")
            log_handle.write(safe_line)
            run_handle.write(safe_line)
        returncode = process.wait()
        log_handle.flush()
        run_handle.flush()
    append_event(
        "command_end",
        name=log_name,
        command=command_text,
        returncode=returncode,
    )
    if returncode != 0:
        raise RuntimeError(
            f"명령 실패(returncode={returncode}). 전체 로그: {log_path}"
        )

def read_secret(name):
    value = os.environ.get(name)
    if value:
        return value
    try:
        return userdata.get(name)
    except Exception:
        return None

GITHUB_TOKEN = read_secret("GITHUB_TOKEN")
if not GITHUB_TOKEN:
    raise RuntimeError("Colab Secrets에 GITHUB_TOKEN을 추가하세요.")
SENSITIVE_VALUES.append(GITHUB_TOKEN)

HF_TOKEN = read_secret("HF_TOKEN")
if HF_TOKEN:
    SENSITIVE_VALUES.append(HF_TOKEN)
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN

# 현재 작업 폴더가 삭제될 저장소 내부여도 안전하게 먼저 빠져나옵니다.
os.chdir("/content")
REPO_DIR = Path("/content/lvlm_insplad")
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)

clone_url = (
    f"https://x-access-token:{GITHUB_TOKEN}"
    "@github.com/arnold6444/lvlm_insplad.git"
)
clone_env = {"GIT_TERMINAL_PROMPT": "0"}
run_cmd(
    [
        "git", "clone", "--branch", REPO_REF, "--single-branch",
        clone_url, REPO_DIR,
    ],
    env=clone_env,
    log_name="setup_clone",
)
run_cmd(
    [
        "git", "-C", REPO_DIR, "remote", "set-url", "origin",
        "https://github.com/arnold6444/lvlm_insplad.git",
    ],
    log_name="setup_clone",
)

run_cmd(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "anomalib==2.6.0",
        "opencv-python-headless>=4.10",
        "scipy>=1.12",
        "ftfy>=6.2",
        "regex>=2024.0",
        "pandas>=2.2",
        "tqdm>=4.66",
        "transformers==5.14.0",
        "accelerate==1.14.0",
        "sentencepiece==0.2.2",
        "pillow>=10",
        "numpy>=1.26",
        "torchsummary",
    ],
    log_name="setup_pip",
)
run_cmd(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "-e", f"{REPO_DIR}[dev]",
    ],
    log_name="setup_pip",
)


# one-shot-plus reference runner도 예측 JSONL을 임시 파일에 쓴 뒤 교체하도록
# 현재 checkout에서만 결정적으로 보강합니다. 중단 중 반쪽 JSONL이 남지 않습니다.
reference_runner_path = REPO_DIR / "scripts/run_hf_aligned_core_full.py"
reference_runner_source = reference_runner_path.read_text(encoding="utf-8")
legacy_write_jsonl = '''def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\\n")
'''
atomic_write_jsonl = '''def write_jsonl(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8") as handle:
        for row in rows:
            handle.write(json.dumps(row, ensure_ascii=False) + "\\n")
        handle.flush()
        try:
            os.fsync(handle.fileno())
        except OSError:
            pass
    temporary.replace(path)
'''
if legacy_write_jsonl in reference_runner_source:
    # os가 기존 runner import에 없으므로 함께 추가합니다.
    reference_runner_source = reference_runner_source.replace(
        "import json\n",
        "import json\nimport os\n",
        1,
    )
    reference_runner_source = reference_runner_source.replace(
        legacy_write_jsonl,
        atomic_write_jsonl,
        1,
    )
    atomic_write_text(reference_runner_path, reference_runner_source)
elif (
    'temporary = path.with_suffix(path.suffix + ".tmp")'
    not in reference_runner_source
):
    raise RuntimeError(
        "reference runner의 write_jsonl 구조가 바뀌어 atomic patch를 "
        "안전하게 적용할 수 없습니다."
    )

SRC_DIR = str(REPO_DIR / "src")
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)
importlib.invalidate_caches()
import insplad_benchmark

os.chdir(REPO_DIR)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("런타임 유형을 GPU로 변경하세요.")
GPU_NAME = torch.cuda.get_device_name(0)
VRAM_GB = torch.cuda.get_device_properties(0).total_memory / 1024**3
if VRAM_GB < 20:
    raise RuntimeError(
        f"현재 VRAM={VRAM_GB:.1f} GiB입니다. L4 또는 A100을 사용하세요."
    )

REPO_COMMIT = subprocess.check_output(
    ["git", "rev-parse", "HEAD"], text=True
).strip()

run_cmd(
    [
        sys.executable, "-m", "pytest", "-q",
        "tests/test_vision_expert.py",
        "tests/test_hf_vision_expert_smoke_runner.py",
        "tests/test_build_vision_expert_smoke_data.py",
        "tests/test_anomalyclip_compat.py",
        "tests/test_vision_expert_smoke_colab.py",
    ],
    cwd=REPO_DIR,
    log_name="setup_tests",
)

RUN_CONFIG = {
    **EXPERIMENT_IDENTITY,
    "experiment_hash": EXPERIMENT_HASH,
    "repo_ref": REPO_REF,
    "repo_commit_sessions": [REPO_COMMIT],
    "result_root": str(RUN_ROOT),
    "created_or_updated_at_utc": utc_now(),
}
previous = read_json(RUN_ROOT / "run_config_resumable.json", {})
previous_commits = list(previous.get("repo_commit_sessions", []))
RUN_CONFIG["repo_commit_sessions"] = list(
    dict.fromkeys([*previous_commits, REPO_COMMIT])
)[-20:]
atomic_write_json(RUN_ROOT / "run_config_resumable.json", RUN_CONFIG)

environment = {
    "session_id": SESSION_ID,
    "time_utc": utc_now(),
    "python": sys.version,
    "repo_commit": REPO_COMMIT,
    "gpu": GPU_NAME,
    "vram_gib": round(VRAM_GB, 3),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "anomalib": importlib.metadata.version("anomalib"),
    "transformers": importlib.metadata.version("transformers"),
}
atomic_write_json(
    RUN_ROOT / "environment" / f"{SESSION_ID}.json",
    environment,
)
mark_stage("setup", "complete", repo_commit=REPO_COMMIT, gpu=GPU_NAME)

print("RUN ROOT:", RUN_ROOT)
print("repo commit:", REPO_COMMIT)
print("GPU:", GPU_NAME, f"{VRAM_GB:.1f} GiB")
print("중단되면 새 런타임에서 다시 모두 실행하세요.")

In [ ]:
# 2) 시작 시 현재 저장 상태 확인
import pandas as pd

def _safe_first_line(value, limit=160):
    if value is None:
        return ""
    text = str(value)
    first_line = text.splitlines()[0] if text.splitlines() else ""
    return first_line[:limit]

def stage_table():
    columns = [
        "stage", "status", "reused", "updated_at_utc", "error",
    ]
    stages = STATE.get("stages", {})
    if not isinstance(stages, dict):
        stages = {}

    rows = []
    for name, raw_value in sorted(
        stages.items(), key=lambda item: str(item[0])
    ):
        value = (
  raw_value
  if isinstance(raw_value, dict)
  else {"status": "invalid_state", "error": raw_value}
        )
        rows.append({
  "stage": str(name),
  "status": value.get("status", ""),
  "reused": value.get("reused", ""),
  "updated_at_utc": value.get("updated_at_utc", ""),
  "error": _safe_first_line(value.get("error", "")),
        })
    return pd.DataFrame(rows, columns=columns)

display(stage_table())
print("상태 파일:", STATE_PATH)
print("통합 로그:", RUN_LOG_PATH)


In [ ]:
# 3) Aligned Core QA + DINOv2 reference + persistent workspace
from zipfile import ZipFile

def resolve_zip(filename):
    if DRIVE_DATA_DIR:
        path = DRIVE_ROOT / DRIVE_DATA_DIR / filename
        if not path.is_file():
            raise FileNotFoundError(path)
        return path
    matches = sorted(DRIVE_ROOT.rglob(filename))
    if not matches:
        local = Path("/content") / filename
        if local.is_file():
            return local
        raise FileNotFoundError(
            f"{filename}을 MyDrive 또는 /content에서 찾지 못했습니다."
        )
    if len(matches) > 1:
        raise RuntimeError(
            f"{filename}이 여러 개입니다. DRIVE_DATA_DIR을 지정하세요: "
            f"{matches[:10]}"
        )
    return matches[0]

unsup_zip = resolve_zip("unsupervised_anomaly_detection.zip")
sup_zip = resolve_zip("supervised_fault_classification.zip")

DATA_ROOT = RUN_ROOT / "data"
MANIFEST_ROOT = DATA_ROOT / "manifests"
BENCH_ROOT = DATA_ROOT / "benchmark"
QA_DIR = DATA_ROOT / "aligned_qa"
REFERENCE_CACHE_ROOT = (
    DRIVE_ROOT / "lvlm_insplad_cache" /
    "dinov2_f9e44c814b77203eaa57a6bdbbd535f21ede1415"
)
PERSISTENT_WORKSPACE_ROOT = RUN_ROOT / "workspace"
LOCAL_WORKSPACE_ROOT = Path("/content/vision_expert_workspace")

for path in [
    DATA_ROOT, MANIFEST_ROOT, QA_DIR,
    REFERENCE_CACHE_ROOT, PERSISTENT_WORKSPACE_ROOT,
]:
    path.mkdir(parents=True, exist_ok=True)

unsup_manifest = MANIFEST_ROOT / "unsup.csv"
sup_manifest = MANIFEST_ROOT / "sup.csv"
SMOKE_QA = QA_DIR / f"one_shot_plus_smoke_{SMOKE_SIZE}.jsonl"

def read_jsonl(path):
    path = Path(path)
    if not path.is_file():
        return []
    rows = []
    for number, line in enumerate(
        path.read_text(encoding="utf-8").splitlines(), start=1
    ):
        if not line.strip():
            continue
        try:
            rows.append(json.loads(line))
        except Exception as exc:
            raise RuntimeError(
                f"손상된 JSONL: {path}, line={number}"
            ) from exc
    return rows

def write_jsonl_atomic(path, rows):
    text = "".join(
        json.dumps(dict(row), ensure_ascii=False) + "\n"
        for row in rows
    )
    atomic_write_text(path, text)

def csv_row_count(path):
    path = Path(path)
    if not path.is_file() or path.stat().st_size == 0:
        return 0
    with path.open(newline="", encoding="utf-8-sig") as handle:
        return sum(1 for _ in csv.DictReader(handle))

def prepare_qa_valid():
    aligned = BENCH_ROOT / "reports/aligned_core.csv"
    full = QA_DIR / "one_shot_plus.jsonl"
    if csv_row_count(aligned) != 1378:
        return False
    smoke_rows_check = read_jsonl(SMOKE_QA)
    if len(smoke_rows_check) != SMOKE_SIZE:
        return False
    ids = [str(row.get("id", "")) for row in smoke_rows_check]
    if not all(ids) or len(ids) != len(set(ids)):
        return False
    required = {
        "image", "reference_image", "answer",
        "asset_category", "retrieval_method",
    }
    if any(not required.issubset(row) for row in smoke_rows_check):
        return False
    return (
        len(read_jsonl(full)) == 1378
        and (REFERENCE_CACHE_ROOT / "reference_embeddings.npz").is_file()
        and (REFERENCE_CACHE_ROOT / "one_shot_plus_references.csv").is_file()
    )

def prepare_qa():
    MANIFEST_ROOT.mkdir(parents=True, exist_ok=True)
    for zip_path, manifest, key in (
        (unsup_zip, unsup_manifest, "unsup"),
        (sup_zip, sup_manifest, "sup"),
    ):
        if not manifest.is_file() or manifest.stat().st_size == 0:
            run_cmd(
                [
                    sys.executable,
                    REPO_DIR / "scripts/inspect_zip_manifest.py",
                    zip_path,
                    "--out-csv", manifest,
                    "--out-json", MANIFEST_ROOT / f"{key}.json",
                ],
                cwd=REPO_DIR,
                log_name=f"prepare_manifest_{key}",
            )

    aligned = BENCH_ROOT / "reports/aligned_core.csv"
    if csv_row_count(aligned) != 1378:
        if BENCH_ROOT.exists():
            shutil.rmtree(BENCH_ROOT)
        run_cmd(
            [
                sys.executable,
                REPO_DIR / "scripts/build_benchmark.py",
                "--anomaly-manifest", unsup_manifest,
                "--fault-manifest", sup_manifest,
                "--overlap-summary",
                REPO_DIR / "reports/image_overlap_summary.json",
                "--repo-root", BENCH_ROOT,
                "--seed", SEED,
            ],
            cwd=REPO_DIR,
            log_name="prepare_benchmark",
        )

    with aligned.open(newline="", encoding="utf-8-sig") as handle:
        core = list(csv.DictReader(handle))
    if len(core) != 1378:
        raise RuntimeError(f"Aligned Core가 1,378장이 아닙니다: {len(core)}")

    zero_by_id = {
        row["id"]: row
        for row in read_jsonl(
            BENCH_ROOT / "data/qa/anomaly_zero_shot.jsonl"
        )
    }
    fixed_by_id = {
        row["id"]: row
        for row in read_jsonl(
            BENCH_ROOT / "data/qa/anomaly_one_shot.jsonl"
        )
    }
    structured_by_id = {
        row["id"]: row
        for row in read_jsonl(
            BENCH_ROOT / "data/qa/defect_qa.jsonl"
        )
    }
    zero_rows = [zero_by_id[row["anomaly_id"]] for row in core]
    fixed_rows = [fixed_by_id[row["anomaly_id"]] for row in core]
    structured_rows = [
        structured_by_id[row["structured_id"]] for row in core
    ]
    write_jsonl_atomic(QA_DIR / "zero_shot.jsonl", zero_rows)
    write_jsonl_atomic(QA_DIR / "one_shot.jsonl", fixed_rows)
    write_jsonl_atomic(QA_DIR / "structured.jsonl", structured_rows)

    run_cmd(
        [
            sys.executable,
            REPO_DIR / "scripts/build_one_shot_plus.py",
            "--anomaly-manifest", unsup_manifest,
            "--unsup-zip", unsup_zip,
            "--base-one-shot-qa", QA_DIR / "one_shot.jsonl",
            "--out-qa", QA_DIR / "one_shot_plus.jsonl",
            "--out-manifest",
            REFERENCE_CACHE_ROOT / "one_shot_plus_references.csv",
            "--cache",
            REFERENCE_CACHE_ROOT / "reference_embeddings.npz",
            "--smoke-out", SMOKE_QA,
            "--smoke-size", SMOKE_SIZE,
            "--model-id", "facebook/dinov2-base",
            "--model-revision",
            "f9e44c814b77203eaa57a6bdbbd535f21ede1415",
            "--device", "cuda",
            "--batch-size", "32",
        ],
        cwd=REPO_DIR,
        log_name="prepare_dinov2_reference",
    )
    run_cmd(
        [
            sys.executable,
            REPO_DIR / "scripts/validate_qa.py",
            QA_DIR / "zero_shot.jsonl",
            QA_DIR / "one_shot_plus.jsonl",
            "--anomaly-manifest", unsup_manifest,
            "--report",
            REFERENCE_CACHE_ROOT / "one_shot_plus_validation.md",
        ],
        cwd=REPO_DIR,
        log_name="prepare_validate_qa",
    )

run_stage("prepare_qa", prepare_qa, prepare_qa_valid)

smoke_rows = read_jsonl(SMOKE_QA)
SMOKE_IDS = [str(row["id"]) for row in smoke_rows]
assert len(smoke_rows) == SMOKE_SIZE
assert len(SMOKE_IDS) == len(set(SMOKE_IDS))

def workspace_valid(root=PERSISTENT_WORKSPACE_ROOT):
    root = Path(root)
    config_path = root / "workspace_config.json"
    manifest_path = root / "workspace_manifest.csv"
    if not config_path.is_file() or not manifest_path.is_file():
        return False
    try:
        config = json.loads(config_path.read_text(encoding="utf-8"))
    except Exception:
        return False
    if int(config.get("qa_count", -1)) != SMOKE_SIZE:
        return False
    with manifest_path.open(newline="", encoding="utf-8-sig") as handle:
        rows = list(csv.DictReader(handle))
    queries = [row for row in rows if row.get("role") == "query"]
    normals = [row for row in rows if row.get("role") == "train_normal"]
    if len(queries) != SMOKE_SIZE or not normals:
        return False
    if {row.get("query_id") for row in queries} != set(SMOKE_IDS):
        return False
    for row in rows:
        path = root / str(row.get("local_path", ""))
        if not path.is_file() or path.stat().st_size == 0:
            return False
    return True

def prepare_workspace():
    command = [
        sys.executable,
        REPO_DIR / "scripts/build_vision_expert_smoke_data.py",
        "--anomaly-manifest", unsup_manifest,
        "--unsup-zip", unsup_zip,
        "--qa", SMOKE_QA,
        "--out-dir", PERSISTENT_WORKSPACE_ROOT,
        "--max-train-per-asset",
        MAX_TRAIN_NORMALS_PER_ASSET,
        "--seed", SEED,
    ]
    if "prepare_workspace" in FORCE_STAGES:
        command.append("--force")
    run_cmd(
        command,
        cwd=REPO_DIR,
        log_name="prepare_workspace",
    )

run_stage(
    "prepare_workspace",
    prepare_workspace,
    lambda: workspace_valid(PERSISTENT_WORKSPACE_ROOT),
)

# Drive의 persistent workspace를 빠른 로컬 SSD로 복사합니다.
# 매 새 런타임에서 이 복사만 다시 하며, 실험 결과는 전부 Drive에 남습니다.
persistent_config_hash = hashlib.sha256(
    (PERSISTENT_WORKSPACE_ROOT / "workspace_config.json").read_bytes()
).hexdigest()
local_marker = LOCAL_WORKSPACE_ROOT / ".persistent_config_sha256"
local_ready = (
    workspace_valid(LOCAL_WORKSPACE_ROOT)
    and local_marker.is_file()
    and local_marker.read_text(encoding="utf-8").strip()
    == persistent_config_hash
)
if not local_ready:
    if LOCAL_WORKSPACE_ROOT.exists():
        shutil.rmtree(LOCAL_WORKSPACE_ROOT)
    print("persistent workspace → Colab 로컬 SSD 복사")
    shutil.copytree(PERSISTENT_WORKSPACE_ROOT, LOCAL_WORKSPACE_ROOT)
    atomic_write_text(local_marker, persistent_config_hash + "\n")

if not workspace_valid(LOCAL_WORKSPACE_ROOT):
    raise RuntimeError("로컬 workspace 복사 검증에 실패했습니다.")

workspace = pd.read_csv(
    LOCAL_WORKSPACE_ROOT / "workspace_manifest.csv"
).fillna("")
query_workspace = workspace.loc[workspace.role == "query"].copy()
train_workspace = workspace.loc[
    workspace.role == "train_normal"
].copy()

display(
    workspace.groupby(["role", "asset_category"])
    .size().rename("n").reset_index()
)
print("Persistent workspace:", PERSISTENT_WORKSPACE_ROOT)
print("Local workspace:", LOCAL_WORKSPACE_ROOT)

In [ ]:
# 4) ROI 공통 helper — ROI 한 장마다 manifest를 원자적으로 저장
import numpy as np
from PIL import Image

from insplad_benchmark.vision_expert import (
    read_manifest,
    resolve_artifact_path,
    save_roi_artifacts,
    validate_roi_manifest,
    write_manifest,
)

EXPERT_ARTIFACT_ROOT = RUN_ROOT / "vision_experts"
ROI_MANIFEST = EXPERT_ARTIFACT_ROOT / "roi_manifest.csv"
EXPERT_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

def read_roi_rows():
    return read_manifest(ROI_MANIFEST)

def write_roi_rows_atomic(rows):
    rows = list(rows)
    tmp = ROI_MANIFEST.with_name(
        f".{ROI_MANIFEST.name}.{uuid.uuid4().hex}.tmp"
    )
    write_manifest(tmp, rows)
    tmp.replace(ROI_MANIFEST)

def roi_row_files_valid(row):
    for field in ("roi_path", "anomaly_map_path", "overlay_path"):
        value = str(row.get(field, ""))
        if not value:
            return False
        path = resolve_artifact_path(
            ROI_MANIFEST, value, artifact_root=RUN_ROOT
        )
        if not path.is_file() or path.stat().st_size == 0:
            return False
    return True

def valid_roi_index():
    index = {}
    for row in read_roi_rows():
        key = (
            str(row.get("query_id", "")),
            str(row.get("expert", "")),
        )
        if all(key) and roi_row_files_valid(row):
            index[key] = row
    return index

def upsert_roi_row(row):
    rows = read_roi_rows()
    key = (str(row["query_id"]), str(row["expert"]))
    kept = [
        existing for existing in rows
        if (
            str(existing.get("query_id", "")),
            str(existing.get("expert", "")),
        ) != key
    ]
    write_roi_rows_atomic([*kept, dict(row)])
    append_event(
        "roi_checkpoint",
        query_id=key[0],
        expert=key[1],
        roi_path=str(row.get("roi_path", "")),
    )

def expert_complete(expert):
    index = valid_roi_index()
    return all((query_id, expert) in index for query_id in SMOKE_IDS)

def pending_ids(expert):
    index = valid_roi_index()
    return [
        query_id for query_id in SMOKE_IDS
        if (query_id, expert) not in index
    ]

def reset_expert(expert):
    rows = [
        row for row in read_roi_rows()
        if str(row.get("expert", "")) != expert
    ]
    write_roi_rows_atomic(rows)
    expert_dir = EXPERT_ARTIFACT_ROOT / expert
    if expert_dir.exists():
        shutil.rmtree(expert_dir)
    append_event("expert_reset", expert=expert)
    print(f"[FORCE] {expert} 기존 ROI artifact를 삭제했습니다.")

def batch_field(batch, name):
    if isinstance(batch, dict):
        return batch[name]
    return getattr(batch, name)

def flatten_anomalib_predictions(predictions):
    records = []
    for batch in predictions:
        paths = batch_field(batch, "image_path")
        maps = batch_field(batch, "anomaly_map")
        try:
            scores = batch_field(batch, "pred_score")
        except Exception:
            scores = None

        if isinstance(paths, (str, Path)):
            paths = [str(paths)]
        else:
            paths = [str(path) for path in paths]

        if torch.is_tensor(maps):
            maps = maps.detach().cpu().numpy()
        maps = np.asarray(maps)
        if maps.ndim == 2:
            maps = maps[None, ...]
        if maps.ndim == 4 and maps.shape[1] == 1:
            maps = maps[:, 0]

        if torch.is_tensor(scores):
            scores = scores.detach().cpu().numpy()
        score_values = (
            [None] * len(paths)
            if scores is None
            else list(np.asarray(scores).reshape(-1))
        )
        for path, anomaly_map, score in zip(
            paths, maps, score_values, strict=True
        ):
            records.append({
                "image_path": Path(path),
                "anomaly_map": anomaly_map,
                "pred_score": (
                    None if score is None else float(score)
                ),
            })
    return records

def release_gpu(*objects):
    for obj in objects:
        try:
            del obj
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.ipc_collect()
        except Exception:
            pass

query_by_resolved_path = {}
query_by_name = {}
query_by_id = {}
for row in query_workspace.itertuples(index=False):
    local_path = (
        LOCAL_WORKSPACE_ROOT / str(row.local_path)
    ).resolve()
    query_by_resolved_path[local_path] = row
    query_by_name.setdefault(local_path.name, []).append(row)
    query_by_id[str(row.query_id)] = row

def resolve_query_row(predicted_path):
    path = Path(predicted_path).resolve()
    row = query_by_resolved_path.get(path)
    if row is not None:
        return row
    matches = query_by_name.get(path.name, [])
    if len(matches) == 1:
        return matches[0]
    raise KeyError(f"prediction image를 query에 매칭하지 못했습니다: {path}")

def save_prediction_records(expert, records, metadata):
    existing = valid_roi_index()
    saved = 0
    for record in records:
        query = resolve_query_row(record["image_path"])
        query_id = str(query.query_id)
        if (query_id, expert) in existing:
            continue
        row = save_roi_artifacts(
            image_path=(
                LOCAL_WORKSPACE_ROOT / str(query.local_path)
            ).resolve(),
            anomaly_map=record["anomaly_map"],
            out_dir=EXPERT_ARTIFACT_ROOT / expert,
            query_id=query_id,
            expert=expert,
            asset_category=str(query.asset_category),
            query_image=str(query.source_path),
            metadata={
                **dict(metadata),
                "model_pred_score": record.get("pred_score", ""),
                "session_id": SESSION_ID,
            },
            artifact_root=RUN_ROOT,
        )
        upsert_roi_row(row)
        existing[(query_id, expert)] = row
        saved += 1
        print(
            f"{expert}: ROI 저장 {saved} | "
            f"남은 수={len(pending_ids(expert))}"
        )

def validate_selected_experts():
    selected_rows = [
        row for row in read_roi_rows()
        if str(row.get("expert", "")) in set(EXPERTS_TO_RUN)
    ]
    validate_roi_manifest(
        selected_rows,
        query_ids=SMOKE_IDS,
        experts=EXPERTS_TO_RUN,
        manifest_path=ROI_MANIFEST,
        artifact_root=RUN_ROOT,
        require_files=True,
    )
    return True

print({
    expert: {
        "complete": expert_complete(expert),
        "pending": len(pending_ids(expert)),
    }
    for expert in EXPERTS_TO_RUN
})

In [ ]:
# 5) Vision expert A — PatchCore
def run_patchcore():
    if "patchcore" not in EXPERTS_TO_RUN:
        return
    if "patchcore_roi" in FORCE_STAGES:
        reset_expert("patchcore")

    from anomalib.data import Tabular
    from anomalib.data.utils import TestSplitMode, ValSplitMode
    from anomalib.engine import Engine
    from anomalib.models import Patchcore
    from lightning import seed_everything

    seed_everything(SEED, workers=True)

    for asset in sorted(query_workspace.asset_category.unique()):
        query_asset = query_workspace.loc[
            query_workspace.asset_category == asset
        ].copy()
        asset_ids = [str(value) for value in query_asset.query_id]
        current_index = valid_roi_index()
        if all((query_id, "patchcore") in current_index for query_id in asset_ids):
            print(f"[RESUME] PatchCore {asset}: 완료된 asset → 건너뜀")
            continue

        train_asset = train_workspace.loc[
            train_workspace.asset_category == asset
        ].copy()
        if train_asset.empty or query_asset.empty:
            raise RuntimeError(
                f"PatchCore data가 비었습니다: {asset}"
            )

        combined = pd.concat(
            [train_asset, query_asset],
            ignore_index=True,
        )
        samples = pd.DataFrame({
            "image_path": [
                str(
                    (
                        LOCAL_WORKSPACE_ROOT /
                        str(path)
                    ).resolve()
                )
                for path in combined.local_path
            ],
            "label_index": [
                int(value) for value in combined.label_index
            ],
            "split": [str(value) for value in combined.split],
        })
        datamodule = Tabular(
            name=f"insplad-patchcore-{asset}",
            samples=samples,
            train_batch_size=16,
            eval_batch_size=8,
            num_workers=2,
            test_split_mode=TestSplitMode.FROM_DIR,
            val_split_mode=ValSplitMode.NONE,
            seed=SEED,
        )
        model = Patchcore(
            backbone="wide_resnet50_2",
            layers=("layer2", "layer3"),
            coreset_sampling_ratio=PATCHCORE_CORESET_RATIO,
            num_neighbors=9,
            post_processor=False,
            evaluator=False,
            visualizer=False,
        )
        engine = Engine(
            accelerator="gpu",
            devices=1,
            max_epochs=1,
            deterministic=True,
            barebones=True,
            limit_val_batches=0,
            default_root_dir=(
                Path("/content/anomalib_runs/patchcore") / asset
            ),
        )

        print(
            f"[PatchCore] {asset}: "
            f"train={len(train_asset)}, query={len(query_asset)}"
        )
        engine.fit(model=model, datamodule=datamodule)
        predictions = engine.predict(
            model=model,
            datamodule=datamodule,
        )
        records = flatten_anomalib_predictions(predictions)
        save_prediction_records(
            "patchcore",
            records,
            {
                "expert_protocol":
                "same-asset leakage-safe train-normal PatchCore",
                "model_id":
                "anomalib.Patchcore/wide_resnet50_2",
                "model_revision": "anomalib==2.6.0",
                "coreset_sampling_ratio":
                PATCHCORE_CORESET_RATIO,
                "train_normal_cap_per_asset":
                MAX_TRAIN_NORMALS_PER_ASSET,
                "asset_model": str(asset),
            },
        )
        if not all(
            (query_id, "patchcore") in valid_roi_index()
            for query_id in asset_ids
        ):
            raise RuntimeError(
                f"PatchCore {asset} ROI가 완성되지 않았습니다."
            )
        release_gpu(predictions, engine, model, datamodule)
        predictions = engine = model = datamodule = None

run_stage(
    "patchcore_roi",
    run_patchcore,
    lambda: (
        "patchcore" not in EXPERTS_TO_RUN
        or expert_complete("patchcore")
    ),
)

In [ ]:
# 6) Vision expert B — Dinomaly
def run_dinomaly():
    if "dinomaly" not in EXPERTS_TO_RUN:
        return
    if "dinomaly_roi" in FORCE_STAGES:
        reset_expert("dinomaly")

    from anomalib.data import Tabular
    from anomalib.data.utils import TestSplitMode, ValSplitMode
    from anomalib.engine import Engine
    from anomalib.models import Dinomaly
    from lightning import seed_everything

    seed_everything(SEED, workers=True)
    combined = pd.concat(
        [train_workspace, query_workspace],
        ignore_index=True,
    )
    samples = pd.DataFrame({
        "image_path": [
            str(
                (
                    LOCAL_WORKSPACE_ROOT /
                    str(path)
                ).resolve()
            )
            for path in combined.local_path
        ],
        "label_index": [
            int(value) for value in combined.label_index
        ],
        "split": [str(value) for value in combined.split],
    })

    last_oom = None
    records = None
    used_batch_size = None

    for batch_size in DINOMALY_BATCH_CANDIDATES:
        predictions = engine = model = datamodule = None
        try:
            print(
                f"[Dinomaly] batch={batch_size}, "
                f"train={len(train_workspace)}, "
                f"query={len(query_workspace)}"
            )
            datamodule = Tabular(
                name="insplad-dinomaly-multiclass",
                samples=samples,
                train_batch_size=batch_size,
                eval_batch_size=max(1, batch_size // 2),
                num_workers=2,
                test_split_mode=TestSplitMode.FROM_DIR,
                val_split_mode=ValSplitMode.NONE,
                seed=SEED,
            )
            model = Dinomaly(
                encoder_name=DINOMALY_ENCODER,
                decoder_depth=DINOMALY_DECODER_DEPTH,
                bottleneck_dropout=0.2,
                use_context_recentering=True,
                precision="float16",
                post_processor=False,
                evaluator=False,
                visualizer=False,
            )
            engine = Engine(
                accelerator="gpu",
                devices=1,
                precision="bf16-mixed",
                max_steps=DINOMALY_MAX_STEPS,
                max_epochs=-1,
                deterministic=True,
                barebones=True,
                limit_val_batches=0,
                default_root_dir="/content/anomalib_runs/dinomaly",
            )
            engine.fit(model=model, datamodule=datamodule)
            predictions = engine.predict(
                model=model,
                datamodule=datamodule,
            )
            records = flatten_anomalib_predictions(predictions)
            used_batch_size = batch_size
            break
        except torch.cuda.OutOfMemoryError as exc:
            last_oom = exc
            append_event(
                "dinomaly_oom",
                batch_size=batch_size,
                error=str(exc),
            )
            print(
                f"Dinomaly OOM: batch={batch_size}. "
                "GPU 메모리를 비우고 더 작은 batch로 재시도합니다."
            )
            release_gpu(predictions, engine, model, datamodule)
            continue

    if records is None:
        raise RuntimeError(
            "Dinomaly가 모든 batch 후보에서 OOM이 발생했습니다."
        ) from last_oom

    save_prediction_records(
        "dinomaly",
        records,
        {
            "expert_protocol":
            "unified multi-asset leakage-safe train-normal Dinomaly",
            "model_id": f"anomalib.Dinomaly/{DINOMALY_ENCODER}",
            "model_revision": "anomalib==2.6.0",
            "decoder_depth": DINOMALY_DECODER_DEPTH,
            "max_steps": DINOMALY_MAX_STEPS,
            "train_batch_size": used_batch_size,
            "use_context_recentering": True,
            "train_normal_cap_per_asset":
            MAX_TRAIN_NORMALS_PER_ASSET,
        },
    )
    release_gpu(predictions, engine, model, datamodule)
    predictions = engine = model = datamodule = None

run_stage(
    "dinomaly_roi",
    run_dinomaly,
    lambda: (
        "dinomaly" not in EXPERTS_TO_RUN
        or expert_complete("dinomaly")
    ),
)

In [ ]:
# 7) Vision expert C — AnomalyCLIP
def ensure_anomalyclip_checkout():
    commit = "3911738c0867544f545a076ad78f3f11d9ecbfdf"
    checkout = Path("/content/AnomalyCLIP")

    valid_checkout = False
    if checkout.is_dir() and (checkout / ".git").is_dir():
        try:
            current = subprocess.check_output(
                ["git", "-C", checkout, "rev-parse", "HEAD"],
                text=True,
            ).strip()
            valid_checkout = current == commit
        except Exception:
            valid_checkout = False

    if not valid_checkout:
        if checkout.exists():
            shutil.rmtree(checkout)
        run_cmd(
            [
                "git", "clone", "--filter=blob:none", "--no-checkout",
                "https://github.com/zqhang/AnomalyCLIP.git",
                checkout,
            ],
            log_name="anomalyclip_checkout",
        )
        run_cmd(
            [
                "git", "-C", checkout,
                "sparse-checkout", "init", "--no-cone",
            ],
            log_name="anomalyclip_checkout",
        )
        sparse_path = (
            checkout / ".git/info/sparse-checkout"
        )
        atomic_write_text(
            sparse_path,
            "/*.py\n"
            "/AnomalyCLIP_lib/\n"
            "/checkpoints/9_12_4_multiscale/epoch_15.pth\n",
        )
        run_cmd(
            [
                "git", "-C", checkout,
                "fetch", "--depth", "1", "origin", commit,
            ],
            log_name="anomalyclip_checkout",
        )
        run_cmd(
            [
                "git", "-C", checkout,
                "checkout", "--detach", commit,
            ],
            log_name="anomalyclip_checkout",
        )

    actual = subprocess.check_output(
        ["git", "-C", checkout, "rev-parse", "HEAD"],
        text=True,
    ).strip()
    if actual != commit:
        raise RuntimeError(
            f"AnomalyCLIP commit 불일치: {actual}"
        )

    from insplad_benchmark.anomalyclip_compat import (
        patch_official_anomalyclip_checkout,
    )
    compat = patch_official_anomalyclip_checkout(checkout)
    return checkout, commit, compat

def run_anomalyclip():
    if "anomalyclip" not in EXPERTS_TO_RUN:
        return
    if "anomalyclip_roi" in FORCE_STAGES:
        reset_expert("anomalyclip")

    checkout, commit, compat = ensure_anomalyclip_checkout()
    for module_name in list(sys.modules):
        if (
            module_name == "AnomalyCLIP_lib"
            or module_name.startswith("AnomalyCLIP_lib.")
            or module_name in {"prompt_ensemble", "utils"}
        ):
            sys.modules.pop(module_name, None)
    if str(checkout) not in sys.path:
        sys.path.insert(0, str(checkout))

    import torch.nn.functional as F
    from scipy.ndimage import gaussian_filter
    from types import SimpleNamespace

    import AnomalyCLIP_lib
    from prompt_ensemble import AnomalyCLIP_PromptLearner
    from utils import get_transform

    ANOMALYCLIP_MODEL_CACHE = (
        RUN_ROOT / "model_cache/anomalyclip"
    )
    ANOMALYCLIP_MODEL_CACHE.mkdir(parents=True, exist_ok=True)

    args = SimpleNamespace(
        image_size=518,
        features_list=[6, 12, 18, 24],
        feature_map_layer=[0, 1, 2, 3],
        depth=9,
        n_ctx=12,
        t_n_ctx=4,
        sigma=4,
    )
    design = {
        "Prompt_length": 12,
        "learnabel_text_embedding_depth": 9,
        "learnabel_text_embedding_length": 4,
    }
    device = "cuda"

    model, _ = AnomalyCLIP_lib.load(
        "ViT-L/14@336px",
        device=device,
        design_details=design,
        download_root=str(ANOMALYCLIP_MODEL_CACHE),
    )
    model.eval()
    preprocess, _ = get_transform(args)
    prompt_learner = AnomalyCLIP_PromptLearner(
        model.to("cpu"), design
    )
    checkpoint_path = (
        checkout /
        "checkpoints/9_12_4_multiscale/epoch_15.pth"
    )
    checkpoint = torch.load(
        checkpoint_path,
        map_location="cpu",
        weights_only=False,
    )
    prompt_learner.load_state_dict(
        checkpoint["prompt_learner"]
    )
    prompt_learner.to(device)
    model.to(device)
    model.visual.DAPM_replace(DPAM_layer=20)

    prompts, tokenized, compound = prompt_learner(cls_id=None)
    text_features = model.encode_text_learn(
        prompts, tokenized, compound
    ).float()
    text_features = torch.stack(
        torch.chunk(text_features, dim=0, chunks=2),
        dim=1,
    )
    text_features = (
        text_features /
        text_features.norm(dim=-1, keepdim=True)
    )

    for index, query_id in enumerate(SMOKE_IDS, start=1):
        if (query_id, "anomalyclip") in valid_roi_index():
            continue

        query = query_by_id[query_id]
        image_path = (
            LOCAL_WORKSPACE_ROOT / str(query.local_path)
        ).resolve()
        image = preprocess(
            Image.open(image_path).convert("RGB")
        ).reshape(
            1, 3, args.image_size, args.image_size
        ).to(device)

        with torch.inference_mode():
            image_features, patch_features = model.encode_image(
                image,
                args.features_list,
                DPAM_layer=20,
            )
            image_features = (
                image_features /
                image_features.norm(dim=-1, keepdim=True)
            )
            image_logits = (
                100.0 *
                image_features @ text_features[0].transpose(-1, -2)
            )
            image_probability = image_logits.softmax(dim=-1)
            image_anomaly_score = float(
                image_probability[0, 1].detach().cpu()
            )

            anomaly_maps = []
            for feature_index, patch_feature in enumerate(
                patch_features
            ):
                if feature_index < args.feature_map_layer[0]:
                    continue
                patch_feature = (
                    patch_feature /
                    patch_feature.norm(dim=-1, keepdim=True)
                )
                similarity, _ = (
                    AnomalyCLIP_lib.compute_similarity(
                        patch_feature,
                        text_features[0],
                    )
                )
                similarity_map = (
                    AnomalyCLIP_lib.get_similarity_map(
                        similarity[:, 1:, :],
                        args.image_size,
                    )
                )
                anomaly_maps.append(
                    (
                        similarity_map[..., 1] +
                        1.0 -
                        similarity_map[..., 0]
                    ) / 2.0
                )
            anomaly_map = torch.stack(
                anomaly_maps
            ).sum(dim=0)[0].detach().cpu().numpy()
            anomaly_map = gaussian_filter(
                anomaly_map,
                sigma=args.sigma,
            )

        row = save_roi_artifacts(
            image_path=image_path,
            anomaly_map=anomaly_map,
            out_dir=EXPERT_ARTIFACT_ROOT / "anomalyclip",
            query_id=query_id,
            expert="anomalyclip",
            asset_category=str(query.asset_category),
            query_image=str(query.source_path),
            metadata={
                "expert_protocol":
                "official pretrained zero-shot AnomalyCLIP",
                "model_id":
                "zqhang/AnomalyCLIP ViT-L/14@336px",
                "model_revision": commit,
                "compat_patch_schema_version":
                compat["schema_version"],
                "compat_source_sha256":
                compat["sha256_after"],
                "checkpoint":
                "checkpoints/9_12_4_multiscale/epoch_15.pth",
                "checkpoint_training_source": "VisA",
                "model_pred_score": image_anomaly_score,
                "session_id": SESSION_ID,
            },
            artifact_root=RUN_ROOT,
        )
        upsert_roi_row(row)
        print(
            f"AnomalyCLIP {index}/{SMOKE_SIZE} | "
            f"남은 수={len(pending_ids('anomalyclip'))}"
        )

    release_gpu(
        model,
        prompt_learner,
        text_features,
        checkpoint,
    )
    model = prompt_learner = text_features = checkpoint = None

run_stage(
    "anomalyclip_roi",
    run_anomalyclip,
    lambda: (
        "anomalyclip" not in EXPERTS_TO_RUN
        or expert_complete("anomalyclip")
    ),
)

In [ ]:
# 8) ROI 전체 검증 + Drive preview 저장
ROI_PREVIEW = RUN_ROOT / "roi_preview.jpg"

def roi_audit_valid():
    if not validate_selected_experts():
        return False
    return (
        not SHOW_ROI_PREVIEW
        or (
            ROI_PREVIEW.is_file()
            and ROI_PREVIEW.stat().st_size > 0
        )
    )

def create_roi_audit():
    validate_selected_experts()
    if not SHOW_ROI_PREVIEW:
        return

    from PIL import ImageDraw, ImageFont

    preview_ids = SMOKE_IDS[:5]
    thumb_w, thumb_h = 360, 260
    columns = 1 + len(EXPERTS_TO_RUN)
    canvas = Image.new(
        "RGB",
        (columns * thumb_w, len(preview_ids) * thumb_h),
        "white",
    )
    draw = ImageDraw.Draw(canvas)
    roi_index = valid_roi_index()

    for row_number, query_id in enumerate(preview_ids):
        query = query_by_id[query_id]
        source_path = (
            LOCAL_WORKSPACE_ROOT / str(query.local_path)
        ).resolve()
        paths = [("query", source_path)]
        for expert in EXPERTS_TO_RUN:
            row = roi_index[(query_id, expert)]
            roi_path = resolve_artifact_path(
                ROI_MANIFEST,
                str(row["roi_path"]),
                artifact_root=RUN_ROOT,
            )
            paths.append((expert, roi_path))

        for column, (label, path) in enumerate(paths):
            image = Image.open(path).convert("RGB")
            image.thumbnail((thumb_w - 12, thumb_h - 36))
            x = column * thumb_w + (thumb_w - image.width) // 2
            y = row_number * thumb_h + 30
            canvas.paste(image, (x, y))
            draw.text(
                (column * thumb_w + 8, row_number * thumb_h + 7),
                f"{label} | {query_id[:12]}",
                fill="black",
            )

    tmp = ROI_PREVIEW.with_name(
        f".{ROI_PREVIEW.name}.{uuid.uuid4().hex}.tmp.jpg"
    )
    canvas.save(tmp, quality=92)
    tmp.replace(ROI_PREVIEW)

run_stage("roi_audit", create_roi_audit, roi_audit_valid)

display(pd.DataFrame([
    {
        "expert": expert,
        "complete": expert_complete(expert),
        "saved_rows": sum(
            str(row.get("expert", "")) == expert
            for row in read_roi_rows()
        ),
    }
    for expert in EXPERTS_TO_RUN
]))
print("ROI manifest:", ROI_MANIFEST)
print("Preview:", ROI_PREVIEW)

In [ ]:
# 9) Qwen + InternVL — reference / ROI / best ROI+reference
from insplad_benchmark.vision_expert import select_best_expert

REFERENCE_RESULT_DIR = (
    RUN_ROOT / f"retrieved_reference_smoke_{SMOKE_SIZE}"
)
ROI_RESULT_DIR = RUN_ROOT / f"roi_smoke_{SMOKE_SIZE}"
WINNER_PATH = RUN_ROOT / "smoke_winner.json"

def quarantine_corrupt_jsonl(path):
    path = Path(path)
    if not path.is_file():
        return
    try:
        read_jsonl(path)
    except Exception as exc:
        quarantine = path.with_name(
            f"{path.name}.corrupt-{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}"
        )
        path.replace(quarantine)
        append_event(
            "corrupt_jsonl_quarantined",
            original=str(path),
            quarantine=str(quarantine),
            error=f"{type(exc).__name__}: {exc}",
        )
        print("손상된 partial JSONL을 격리했습니다:", quarantine)

def successful_prediction_file(path, expected_ids):
    rows = read_jsonl(path)
    by_id = {str(row.get("id", "")): row for row in rows}
    if set(by_id) != set(expected_ids):
        return False
    return all(
        str(by_id[query_id].get("output", "")).strip()
        and not by_id[query_id].get("error")
        for query_id in expected_ids
    )

def reference_inference_valid():
    if not RUN_REFERENCE:
        return True
    leaderboard = REFERENCE_RESULT_DIR / "leaderboard.csv"
    if not leaderboard.is_file() or leaderboard.stat().st_size == 0:
        return False
    return all(
        successful_prediction_file(
            REFERENCE_RESULT_DIR /
            model_key / "one_shot_plus.jsonl",
            SMOKE_IDS,
        )
        for model_key in MODELS_TO_RUN
    )

def run_reference_inference():
    if not RUN_REFERENCE:
        return
    if "reference_inference" in FORCE_STAGES and REFERENCE_RESULT_DIR.exists():
        shutil.rmtree(REFERENCE_RESULT_DIR)
    for model_key in MODELS_TO_RUN:
        quarantine_corrupt_jsonl(
            REFERENCE_RESULT_DIR / model_key / "one_shot_plus.jsonl"
        )
    run_cmd(
        [
            sys.executable,
            REPO_DIR / "scripts/run_hf_aligned_core_full.py",
            "--experiment", "one_shot_plus",
            "--qa-dir", QA_DIR,
            "--unsup-zip", unsup_zip,
            "--sup-zip", sup_zip,
            "--out-dir", REFERENCE_RESULT_DIR,
            "--smoke-size", SMOKE_SIZE,
            "--checkpoint-every", LVLM_CHECKPOINT_EVERY,
            "--models", *MODELS_TO_RUN,
        ],
        cwd=REPO_DIR,
        log_name="reference_inference",
    )

run_stage(
    "reference_inference",
    run_reference_inference,
    reference_inference_valid,
)

def roi_inference_valid():
    if not RUN_ROI_LVLM:
        return True
    leaderboard = ROI_RESULT_DIR / "leaderboard.csv"
    if not leaderboard.is_file() or leaderboard.stat().st_size == 0:
        return False
    return all(
        successful_prediction_file(
            ROI_RESULT_DIR / "predictions" /
            model_key / f"roi__{expert}.jsonl",
            SMOKE_IDS,
        )
        for model_key in MODELS_TO_RUN
        for expert in EXPERTS_TO_RUN
    )

def run_roi_inference():
    if not RUN_ROI_LVLM:
        return
    if "roi_inference" in FORCE_STAGES:
        for model_key in MODELS_TO_RUN:
            for expert in EXPERTS_TO_RUN:
                path = (
                    ROI_RESULT_DIR / "predictions" / model_key /
                    f"roi__{expert}.jsonl"
                )
                if path.exists():
                    path.unlink()
    for model_key in MODELS_TO_RUN:
        for expert in EXPERTS_TO_RUN:
            quarantine_corrupt_jsonl(
                ROI_RESULT_DIR / "predictions" / model_key /
                f"roi__{expert}.jsonl"
            )
    run_cmd(
        [
            sys.executable,
            REPO_DIR / "scripts/run_hf_vision_expert_smoke.py",
            "--qa", SMOKE_QA,
            "--roi-manifest", ROI_MANIFEST,
            "--artifact-root", RUN_ROOT,
            "--unsup-zip", unsup_zip,
            "--out-dir", ROI_RESULT_DIR,
            "--condition", "roi",
            "--experts", *EXPERTS_TO_RUN,
            "--models", *MODELS_TO_RUN,
            "--checkpoint-every", LVLM_CHECKPOINT_EVERY,
            "--attempts", LVLM_RETRY_ATTEMPTS,
            "--continue-on-model-error",
        ],
        cwd=REPO_DIR,
        log_name="roi_inference",
    )

run_stage(
    "roi_inference",
    run_roi_inference,
    roi_inference_valid,
)

def winner_valid():
    if not WINNER_PATH.is_file():
        return False
    try:
        winner = json.loads(
            WINNER_PATH.read_text(encoding="utf-8")
        )
    except Exception:
        return False
    return (
        str(winner.get("expert", "")) in EXPERTS_TO_RUN
        and winner.get("ranking")
    )

def select_winner():
    if "select_winner" in FORCE_STAGES and WINNER_PATH.exists():
        WINNER_PATH.unlink()
    leaderboard_path = ROI_RESULT_DIR / "leaderboard.csv"
    rows = pd.read_csv(leaderboard_path).to_dict("records")
    winner = select_best_expert(
        rows,
        required_models=MODELS_TO_RUN,
        condition="roi",
    )
    winner.update({
        "selected_at_utc": utc_now(),
        "smoke_size": SMOKE_SIZE,
        "selection_role": "method-selection smoke only",
        "selection_rule":
        "mean F1, then mean balanced accuracy, then invalid rate",
    })
    atomic_write_json(WINNER_PATH, winner)
    print("BEST EXPERT:", winner["expert"])
    display(pd.DataFrame(winner["ranking"]))

run_stage("select_winner", select_winner, winner_valid)
WINNER = json.loads(WINNER_PATH.read_text(encoding="utf-8"))
BEST_EXPERT = str(WINNER["expert"])

def combined_inference_valid():
    if not RUN_COMBINED:
        return True
    leaderboard = ROI_RESULT_DIR / "leaderboard.csv"
    if not leaderboard.is_file() or leaderboard.stat().st_size == 0:
        return False
    return all(
        successful_prediction_file(
            ROI_RESULT_DIR / "predictions" /
            model_key /
            f"roi_plus_reference__{BEST_EXPERT}.jsonl",
            SMOKE_IDS,
        )
        for model_key in MODELS_TO_RUN
    )

def run_combined_inference():
    if not RUN_COMBINED:
        return
    for model_key in MODELS_TO_RUN:
        path = (
            ROI_RESULT_DIR / "predictions" / model_key /
            f"roi_plus_reference__{BEST_EXPERT}.jsonl"
        )
        if "combined_inference" in FORCE_STAGES and path.exists():
            path.unlink()
        quarantine_corrupt_jsonl(path)
    run_cmd(
        [
            sys.executable,
            REPO_DIR / "scripts/run_hf_vision_expert_smoke.py",
            "--qa", SMOKE_QA,
            "--roi-manifest", ROI_MANIFEST,
            "--artifact-root", RUN_ROOT,
            "--unsup-zip", unsup_zip,
            "--out-dir", ROI_RESULT_DIR,
            "--condition", "roi_plus_reference",
            "--experts", BEST_EXPERT,
            "--models", *MODELS_TO_RUN,
            "--checkpoint-every", LVLM_CHECKPOINT_EVERY,
            "--attempts", LVLM_RETRY_ATTEMPTS,
            "--continue-on-model-error",
        ],
        cwd=REPO_DIR,
        log_name="combined_inference",
    )

run_stage(
    "combined_inference",
    run_combined_inference,
    combined_inference_valid,
)

In [ ]:
# 10) 최종 비교표, 상태 요약, 결과 ZIP
SCREENING_COMPARISON = RUN_ROOT / "screening_comparison.csv"
RUN_SUMMARY = RUN_ROOT / "run_summary_resumable.json"

def finalize_valid():
    return (
        SCREENING_COMPARISON.is_file()
        and SCREENING_COMPARISON.stat().st_size > 0
        and RUN_SUMMARY.is_file()
        and WINNER_PATH.is_file()
    )

def finalize():
    comparison_rows = []

    reference_leaderboard = (
        REFERENCE_RESULT_DIR / "leaderboard.csv"
    )
    if reference_leaderboard.is_file():
        reference = pd.read_csv(reference_leaderboard)
        for _, row in reference.iterrows():
            comparison_rows.append({
                "model_key": row["model_key"],
                "condition": "retrieved_reference",
                "expert": "dinov2_retriever",
                "balanced_accuracy":
                row.get("one_shot_plus_bacc", ""),
                "f1": row.get("one_shot_plus_f1", ""),
                "invalid_output_rate":
                row.get("one_shot_plus_invalid", ""),
            })

    roi_leaderboard = ROI_RESULT_DIR / "leaderboard.csv"
    if roi_leaderboard.is_file():
        roi = pd.read_csv(roi_leaderboard)
        keep = [
            column for column in [
                "model_key", "condition", "expert",
                "balanced_accuracy", "f1", "precision", "recall",
                "invalid_output_rate", "tp", "fp", "tn", "fn",
            ]
            if column in roi.columns
        ]
        comparison_rows.extend(
            roi[keep].to_dict("records")
        )

    if not comparison_rows:
        raise RuntimeError("최종 비교할 leaderboard가 없습니다.")

    comparison = pd.DataFrame(comparison_rows).sort_values(
        ["condition", "expert", "model_key"]
    )
    tmp = SCREENING_COMPARISON.with_name(
        f".{SCREENING_COMPARISON.name}.{uuid.uuid4().hex}.tmp"
    )
    comparison.to_csv(tmp, index=False)
    tmp.replace(SCREENING_COMPARISON)

    summary = {
        "schema_version": 2,
        "experiment_hash": EXPERIMENT_HASH,
        "repo_commit": REPO_COMMIT,
        "session_id": SESSION_ID,
        "updated_at_utc": utc_now(),
        "run_root": str(RUN_ROOT),
        "smoke_size": SMOKE_SIZE,
        "models": MODELS_TO_RUN,
        "experts": EXPERTS_TO_RUN,
        "best_expert": BEST_EXPERT,
        "winner": WINNER,
        "state_path": str(STATE_PATH),
        "events_path": str(EVENTS_PATH),
        "run_log_path": str(RUN_LOG_PATH),
        "screening_comparison": str(SCREENING_COMPARISON),
        "resume_instruction":
        "새 Colab 런타임에서 이 notebook을 다시 모두 실행",
        "dinomaly_resume_scope":
        "학습 중단 시 Dinomaly 단계만 재학습; 다른 완료 단계는 재사용",
    }
    atomic_write_json(RUN_SUMMARY, summary)

run_stage("finalize", finalize, finalize_valid)

comparison = pd.read_csv(SCREENING_COMPARISON)
display(comparison)
display(stage_table())

archive_path = None
if SAVE_FULL_RESULT_ZIP_TO_DRIVE or AUTO_DOWNLOAD_RESULT_ZIP:
    archive_base = Path("/content") / (
        f"{RUN_FOLDER}_FULL_result_{EXPERIMENT_HASH[:8]}"
    )
    archive_path = Path(
        shutil.make_archive(
            str(archive_base),
            "zip",
            root_dir=RUN_ROOT,
        )
    )

    if SAVE_FULL_RESULT_ZIP_TO_DRIVE:
        full_export_dir = BASE_RESULT_ROOT / "_exports"
        full_export_dir.mkdir(parents=True, exist_ok=True)
        full_drive_path = full_export_dir / archive_path.name
        full_tmp = full_drive_path.with_name(
            f".{full_drive_path.name}.{uuid.uuid4().hex}.tmp"
        )
        shutil.copy2(archive_path, full_tmp)
        full_tmp.replace(full_drive_path)
        print("FULL RESULT ZIP (Drive):", full_drive_path)

print("RESULT ROOT:", RUN_ROOT)
print("BEST EXPERT:", BEST_EXPERT)
print("다시 실행하면 완료된 결과를 검증한 뒤 자동으로 이어집니다.")

if AUTO_DOWNLOAD_RESULT_ZIP and archive_path is not None:
    from google.colab import files
    files.download(str(archive_path))

In [ ]:
# 11) 결과 확인용 ZIP 생성 — 이 셀만 따로 실행해도 됩니다.
# 완료 결과뿐 아니라 중간 상태/로그도 Google Drive에서 찾아 작은 REVIEW ZIP으로 묶습니다.
import json
import shutil
import uuid
from datetime import datetime, timezone
from pathlib import Path

try:
    from google.colab import drive, files
except ImportError as exc:
    raise RuntimeError("Google Colab에서 실행하세요.") from exc

if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive", force_remount=False)

review_drive_root = Path("/content/drive/MyDrive")
review_base = review_drive_root / "lvlm_insplad_results"
review_run_folder = globals().get("RUN_FOLDER", "vision_expert_smoke_v1")
review_auto_download = bool(globals().get("AUTO_DOWNLOAD_REVIEW_ZIP", True))

candidates = sorted(
    [path for path in review_base.glob(f"{review_run_folder}*") if path.is_dir()],
    key=lambda path: path.stat().st_mtime,
    reverse=True,
)

current_run_root = globals().get("RUN_ROOT")
if current_run_root is not None:
    current_run_root = Path(current_run_root)
    if current_run_root.is_dir() and current_run_root not in candidates:
        candidates.insert(0, current_run_root)

if not candidates:
    raise FileNotFoundError(
        f"결과 폴더를 찾지 못했습니다: {review_base}/{review_run_folder}*"
    )

# 최종 표가 있는 폴더를 최우선, 그다음 상태 파일이 있는 최신 폴더를 선택합니다.
selected_root = next(
    (path for path in candidates if (path / "screening_comparison.csv").is_file()),
    next(
        (path for path in candidates if (path / "run_state.json").is_file()),
        candidates[0],
    ),
)

print("발견된 결과 폴더:")
for path in candidates:
    print("-", path)
print("\n선택된 결과 폴더:", selected_root)

smoke_size = int(globals().get("SMOKE_SIZE", 50))
important_relatives = [
    "run_config_resumable.json",
    "run_state.json",
    "run_summary_resumable.json",
    "screening_comparison.csv",
    "smoke_winner.json",
    "roi_preview.jpg",
    "vision_experts/roi_manifest.csv",
    f"retrieved_reference_smoke_{smoke_size}/leaderboard.csv",
    f"retrieved_reference_smoke_{smoke_size}/run_status.json",
    f"retrieved_reference_smoke_{smoke_size}/run_metadata.json",
    f"roi_smoke_{smoke_size}/leaderboard.csv",
    f"roi_smoke_{smoke_size}/anomaly_by_asset.csv",
    f"roi_smoke_{smoke_size}/prediction_review.csv",
    f"roi_smoke_{smoke_size}/run_status__roi.json",
    f"roi_smoke_{smoke_size}/run_status__roi_plus_reference.json",
    "logs/events.jsonl",
]

print("\n주요 파일 상태:")
status_rows = []
for relative in important_relatives:
    source = selected_root / relative
    exists = source.is_file()
    size = source.stat().st_size if exists else 0
    status_rows.append({"path": relative, "exists": exists, "size_bytes": size})
    print(
        f"- {relative}: {'있음' if exists else '없음'}",
        f"({size:,} bytes)" if exists else "",
    )

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
review_work = Path("/content") / f"vision_expert_smoke_REVIEW_{stamp}"
if review_work.exists():
    shutil.rmtree(review_work)
review_work.mkdir(parents=True)

included = []
missing = []
for relative in important_relatives:
    source = selected_root / relative
    if not source.is_file():
        missing.append(relative)
        continue
    destination = review_work / relative
    destination.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(source, destination)
    included.append(relative)

# 전체 로그는 커질 수 있으므로 마지막 1,000줄만 포함합니다.
run_log = selected_root / "logs/run.log"
if run_log.is_file():
    with run_log.open("r", encoding="utf-8", errors="replace") as handle:
        tail_lines = handle.readlines()[-1000:]
    (review_work / "logs").mkdir(parents=True, exist_ok=True)
    (review_work / "logs/run_log_tail.txt").write_text(
        "".join(tail_lines),
        encoding="utf-8",
    )
    included.append("logs/run_log_tail.txt")

manifest = {
    "schema_version": 1,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "selected_result_root": str(selected_root),
    "candidate_result_roots": [str(path) for path in candidates],
    "included": included,
    "missing": missing,
    "file_status": status_rows,
    "instruction": (
        "이 REVIEW ZIP을 ChatGPT 채팅에 첨부하면 결과를 바로 분석할 수 있습니다."
    ),
}
(review_work / "review_manifest.json").write_text(
    json.dumps(manifest, ensure_ascii=False, indent=2) + "\n",
    encoding="utf-8",
)

local_zip = Path(
    shutil.make_archive(
        str(review_work),
        "zip",
        root_dir=review_work,
    )
)

# 버전별 보관본 + 항상 같은 이름의 최신본을 Drive에 원자적으로 저장합니다.
export_dir = review_base / "_exports"
export_dir.mkdir(parents=True, exist_ok=True)
drive_versioned_zip = export_dir / local_zip.name
drive_latest_zip = review_drive_root / "vision_expert_smoke_REVIEW_LATEST.zip"

for destination in (drive_versioned_zip, drive_latest_zip):
    temporary = destination.with_name(
        f".{destination.name}.{uuid.uuid4().hex}.tmp"
    )
    shutil.copy2(local_zip, temporary)
    temporary.replace(destination)

print("\nREVIEW ZIP (local):", local_zip)
print("REVIEW ZIP (Drive versioned):", drive_versioned_zip)
print("REVIEW ZIP (Drive latest):", drive_latest_zip)
print("ChatGPT에는 vision_expert_smoke_REVIEW_LATEST.zip을 올리면 됩니다.")

if review_auto_download:
    files.download(str(local_zip))


## 중단됐을 때

새 Colab 런타임에서 같은 노트북을 열고 다시 `런타임 → 모두 실행`하면 됩니다.

자동 재사용 단위:

- QA / DINOv2 reference cache: 단계 단위
- workspace: Drive 영구 보관 후 로컬 SSD로 다시 복사
- PatchCore: asset 단위
- AnomalyCLIP: 이미지 단위
- Qwen / InternVL: 예측 1건 단위
- leaderboard / winner / 최종 ZIP: 검증 후 재생성

오류 원문은 다음 위치에 남습니다.

```text
MyDrive/lvlm_insplad_results/vision_expert_smoke_v1/logs/
```

다음 연구 단계:

1. smoke winner를 50장 제외 1,328장 holdout에서 검증
2. winner ROI + reference에 defect RAG 추가
3. 고정 결합과 AgentIAD-lite 비교

결과 확인용 파일:

```text
MyDrive/vision_expert_smoke_REVIEW_LATEST.zip
```

마지막 결과 내보내기 셀은 독립 실행할 수 있습니다. 이 ZIP을 ChatGPT 채팅에 첨부하면 winner, 모델별 성능, 다음 실험을 바로 분석할 수 있습니다.
